In [1]:
# ============================================================
# CODE BLOCK 1: Clean stable setup for Stage 1 notebook
# ============================================================

# Install specific versions of libraries to ensure compatibility and stability.
# Using pinned versions avoids breaking changes from future updates.

# transformers==4.44.2 → Hugging Face Transformers library
# datasets==2.21.0 → Hugging Face Datasets library
# accelerate==0.34.2 → Multi-GPU/TPU training
# huggingface_hub==0.25.2 → Hub client
# evaluate==0.4.2 → Metrics framework
# rouge-score → ROUGE metric
# scikit-learn → ML utilities
# gradio → Interactive UI
# peft → required for LoRA fine-tuning

!pip install -q \
    transformers==4.44.2 \
    datasets==2.21.0 \
    accelerate==0.34.2 \
    huggingface_hub==0.25.2 \
    evaluate==0.4.2 \
    peft==0.13.2 \
    bert-score \
    rouge-score \
    scikit-learn \
    gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.6/436.6 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.9/321.9 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 120.1 MB/s eta 0:00:00
   

In [2]:
# ============================================================
# CODE BLOCK 2: Verify versions and import common libraries
# ============================================================

import os              # Provides functions for interacting with the operating system
                       # (file paths, environment variables, directory operations).

import shutil          # High-level file operations (copy, move, delete directories/files).
                       # Useful for cleaning up output folders between runs.

import torch           # PyTorch library for tensor operations, GPU acceleration,
                       # and deep learning model training.

import numpy as np     # NumPy library for numerical computations, arrays, and
                       # mathematical operations. Often used for metrics and preprocessing.

import pandas as pd    # Pandas library for data manipulation and analysis.
                       # Useful for tabular data inspection or logging results.

import transformers    # Hugging Face Transformers library (models, tokenizers, Trainer).
                       # Core framework for loading and fine-tuning models like CodeGen.

import datasets        # Hugging Face Datasets library for loading and processing datasets.
                       # Provides efficient dataset handling with lazy loading and caching.

import accelerate      # Hugging Face Accelerate library for distributed training,
                       # mixed precision, and hardware abstraction (CPU/GPU/TPU).

import huggingface_hub # Hugging Face Hub client for authentication, uploading models,
                       # and downloading pretrained models/datasets.
import bert_score

from bert_score import score as bert_score

from datasets import load_dataset
# load_dataset is the main function to fetch datasets from Hugging Face Hub.
# Example: load_dataset("code_search_net", "python")

from transformers import AutoTokenizer
# AutoTokenizer automatically loads the correct tokenizer for a given model.
# Tokenizers convert text/code into token IDs that the model can process.

from transformers import AutoModelForCausalLM
# AutoModelForCausalLM loads a causal language model (predicts next token).
# Example: CodeGen, GPT-style models.

from transformers import AutoModel
# AutoModel is a generic loader for models without specifying task type.
# Useful for embeddings or feature extraction.

from transformers import Trainer
# Trainer is Hugging Face’s high-level training loop abstraction.
# Handles training, evaluation, checkpointing, and logging.

from transformers import TrainingArguments
# TrainingArguments defines hyperparameters and settings for Trainer.
# Includes batch size, learning rate, logging, saving, etc.

from transformers import default_data_collator
# default_data_collator batches examples together and pads them to equal length.
# Ensures consistent input shapes for training.

import evaluate
# Hugging Face Evaluate library for metrics (ROUGE, BLEU, CodeBLEU, BERTScore).
# Provides a unified API for computing evaluation scores.

from datasets import concatenate_datasets
# Used to merge Python and Java datasets into one multilingual training dataset.

from peft import LoraConfig
# LoraConfig defines LoRA hyperparameters such as rank, alpha, dropout,
# and which layers should receive LoRA adapters.

from peft import get_peft_model
# get_peft_model injects LoRA adapter layers into the base CodeGen model.

from peft import TaskType
# TaskType tells PEFT what kind of model/task we are adapting.
# Here we use CAUSAL_LM because CodeGen is a decoder-only causal language model.

# Print version numbers to verify environment setup.
print("torch:", torch.__version__)             # Shows installed PyTorch version.
print("transformers:", transformers.__version__) # Shows Transformers version.
print("datasets:", datasets.__version__)         # Shows Datasets version.
print("accelerate:", accelerate.__version__)     # Shows Accelerate version.
print("huggingface_hub:", huggingface_hub.__version__) # Shows Hub client version.


torch: 2.11.0+cu128
transformers: 4.44.2
datasets: 2.21.0
accelerate: 0.34.2
huggingface_hub: 0.25.2


In [3]:
# ============================================================
# CODE BLOCK 3: Configuration
# ============================================================

MODEL_NAME = "Salesforce/codegen-350M-multi"
# One multilingual CodeGen model is used for both Python and Java.
# This is better than training separate models because:
# 1. CodeGen-350M-Multi was already trained on multiple programming languages.
# 2. Documentation patterns such as purpose, parameters, and return values
#    are shared across languages.
# 3. A single model is easier to deploy and maintain.
# 4. It supports your future project direction of Python ↔ Java code tasks.

OFFICIAL_CODESEARCHNET_NAME = "code_search_net"
# Official Hugging Face CodeSearchNet dataset loader.
# We will try to use this for Java first.

OFFICIAL_PYTHON_CONFIG = "python"
# Official CodeSearchNet Python configuration name.

OFFICIAL_JAVA_CONFIG = "java"
# Official CodeSearchNet Java configuration name.

PYTHON_FALLBACK_DATASET_NAME = "Nan-Do/code-search-net-python"
# Maintained CodeSearchNet-style Python dataset.
# We use this instead of the legacy official Python loader because the old
# code_search_net loader caused compatibility issues earlier.

JAVA_FALLBACK_DATASET_NAME = "Nan-Do/code-search-net-java"
# Fallback maintained Java dataset.
# This is used only if the official Java split fails to load.
# This fallback is still CodeSearchNet-style Java code-docstring data.

OUTPUT_DIR = "/content/stage1_codegen_doc_lora_model"
# Directory where LoRA adapters, checkpoints, and tokenizer are saved.

MAX_LENGTH = 512
# Maximum sequence length for prompt + code + documentation.
# Longer length gives more context but consumes more memory.

DEMO_MODE = True
# True:
#   Use a moderate multilingual subset for quick Colab training.
# False:
#   Use K-sample chunked training for larger non-demo training.

# ------------------------------------------------------------
# Demo-mode sample sizes
# ------------------------------------------------------------

PYTHON_TRAIN_SAMPLES = 2500
JAVA_TRAIN_SAMPLES = 2500
# Demo training uses 2,500 Python + 2,500 Java examples.
# This is much stronger than a tiny 500+500 smoke test,
# but still practical for Colab with LoRA.

PYTHON_EVAL_SAMPLES = 250
JAVA_EVAL_SAMPLES = 250
# Demo evaluation uses 250 examples per language.

# ------------------------------------------------------------
# Non-demo K-sample training configuration
# ------------------------------------------------------------

NON_DEMO_EVAL_PER_LANGUAGE = 2000
# In non-demo mode, reserve 2,000 examples per language for evaluation.
# This gives a stronger evaluation set without making validation too slow.

K_SAMPLE_TRAINING = True
# Enables chunked K-sample training in non-demo mode.
#
# Why chunked training?
# Python alone has hundreds of thousands of examples.
# Java may also be large.
# Training the entire combined dataset in one run can be slow and memory-heavy.
# K-sample training lets us train round-by-round on manageable chunks.

K_SAMPLES_PER_LANGUAGE_PER_ROUND = 10000
# Each non-demo round uses:
# 10,000 Python examples + 10,000 Java examples = 20,000 examples total.

K_TRAINING_ROUNDS = 5
# Total non-demo exposure if both languages are available:
# 5 rounds × 20,000 examples = 100,000 examples.

# ------------------------------------------------------------
# Evaluation configuration
# ------------------------------------------------------------

TEST_EXAMPLES = 100
# Number of examples used for baseline and fine-tuned automatic evaluation.

# ------------------------------------------------------------
# CodeBERTScore length filtering
# ------------------------------------------------------------

MIN_DOC_WORDS_FOR_CODEBERT = 4
# Very short outputs can produce misleadingly high semantic similarity.
# Example: "Returns value" may be too vague but still score high.
# We skip CodeBERTScore when prediction/reference is below this length.

MAX_DOC_WORDS_FOR_CODEBERT = 250
# Very long docs can slow evaluation and dilute embedding meaning.
# We skip CodeBERTScore for excessively long documentation.

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)

Device: cuda


In [4]:
# ============================================================
# CODE BLOCK 4: Load Python + Java documentation datasets
# ============================================================

def standardize_dataset_columns(dataset_split, language_name):
    """
    Standardizes dataset columns so Python and Java examples have the same schema.
    Final schema:
    - code
    - docstring
    - language_tag
    """

    def normalize_example(example):
        # Extract code robustly.
        if "code" in example and example["code"] is not None:
            code = example["code"]
        elif "func_code_string" in example and example["func_code_string"] is not None:
            code = example["func_code_string"]
        elif "original_string" in example and example["original_string"] is not None:
            code = example["original_string"]
        else:
            code = ""

        # Extract documentation robustly.
        if "docstring" in example and example["docstring"] is not None:
            docstring = example["docstring"]
        elif "func_documentation_string" in example and example["func_documentation_string"] is not None:
            docstring = example["func_documentation_string"]
        elif "summary" in example and example["summary"] is not None:
            docstring = example["summary"]
        elif "documentation" in example and example["documentation"] is not None:
            docstring = example["documentation"]
        else:
            docstring = ""

        return {
            "code": code,
            "docstring": docstring,
            "language_tag": language_name
        }

    standardized = dataset_split.map(normalize_example)
    standardized = standardized.select_columns(["code", "docstring", "language_tag"])
    return standardized


def load_python_dataset():
    """
    Loads Python documentation dataset.

    Strategy:
    1. Try official CodeSearchNet Python split first.
    2. If it fails, fallback to maintained Nan-Do Python mirror.
    """

    try:
        print("Trying official CodeSearchNet Python split first...")

        official_python = load_dataset(
            OFFICIAL_CODESEARCHNET_NAME,
            OFFICIAL_PYTHON_CONFIG
        )

        python_train = official_python["train"]

        python_train = standardize_dataset_columns(
            python_train,
            "Python"
        )

        print("Official CodeSearchNet Python split loaded successfully.")
        print(python_train)

        return python_train

    except Exception as official_error:
        print("Official Python loader failed.")
        print("Reason:", official_error)

        print("\nFalling back to maintained Nan-Do Python CodeSearchNet mirror...")

        fallback_python = load_dataset(PYTHON_FALLBACK_DATASET_NAME)

        python_train = fallback_python["train"]

        python_train = standardize_dataset_columns(
            python_train,
            "Python"
        )

        print("Fallback Python dataset loaded successfully.")
        print(python_train)

        return python_train


def load_java_dataset():
    """
    Loads Java documentation dataset.
    Strategy:
    1. Try official CodeSearchNet Java split first.
    2. If it fails, fallback to maintained Nan-Do Java mirror.
    """

    try:
        print("Trying official CodeSearchNet Java split first...")

        official_java = load_dataset(
            OFFICIAL_CODESEARCHNET_NAME,
            OFFICIAL_JAVA_CONFIG
        )

        java_train = official_java["train"]

        java_train = standardize_dataset_columns(
            java_train,
            "Java"
        )

        print("Official CodeSearchNet Java split loaded successfully.")
        print(java_train)

        return java_train

    except Exception as official_error:
        print("Official Java loader failed.")
        print("Reason:", official_error)

        print("\nFalling back to maintained Nan-Do Java CodeSearchNet mirror...")

        fallback_java = load_dataset(JAVA_FALLBACK_DATASET_NAME)

        java_train = fallback_java["train"]

        java_train = standardize_dataset_columns(
            java_train,
            "Java"
        )

        print("Fallback Java dataset loaded successfully.")
        print(java_train)

        return java_train


# Load both datasets
python_dataset = load_python_dataset()
java_dataset = load_java_dataset()

# Combine and shuffle
combined_dataset = concatenate_datasets([python_dataset, java_dataset])
combined_dataset = combined_dataset.shuffle(seed=42)

print("\nCombined multilingual dataset:")
print(combined_dataset)

print("\nCombined columns:")
print(combined_dataset.column_names)

print("\nLanguage distribution:")
print(pd.Series(combined_dataset["language_tag"]).value_counts())


Trying official CodeSearchNet Python split first...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:90: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/412178 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/22176 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/23107 [00:00<?, ? examples/s]

...

Generating train split:   0%|          | 0/454451 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/26909 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/15328 [00:00<?, ? examples/s]

Map:   0%|          | 0/454451 [00:00<?, ? examples/s]

Official CodeSearchNet Java split loaded successfully.
Dataset({
    features: ['code', 'docstring', 'language_tag'],
    num_rows: 454451
})

Combined multilingual dataset:
Dataset({
    features: ['code', 'docstring', 'language_tag'],
    num_rows: 866629
})

Combined columns:
['code', 'docstring', 'language_tag']

Language distribution:
Java      454451
Python    412178
Name: count, dtype: int64


In [5]:
# ============================================================
# CODE BLOCK 5: Multilingual dataset exploration
# ============================================================

print("Combined Dataset Columns:")
print(combined_dataset.column_names)

print("\nPython Sample:")
python_sample = python_dataset[0]
print("Language:", python_sample["language_tag"])
print("Code:\n", python_sample["code"][:1000])
print("\nDocstring:\n", python_sample["docstring"][:500])

print("\nJava Sample:")
java_sample = java_dataset[0]
print("Language:", java_sample["language_tag"])
print("Code:\n", java_sample["code"][:1000])
print("\nDocstring:\n", java_sample["docstring"][:500])

Combined Dataset Columns:
['code', 'docstring', 'language_tag']

Python Sample:
Language: Python
Code:
 def __msgc_step3_discontinuity_localization(self):
        """
        Estimate discontinuity in basis of low resolution image segmentation.
        :return: discontinuity in low resolution
        """
        import scipy

        start = self._start_time
        seg = 1 - self.segmentation.astype(np.int8)
        self.stats["low level object voxels"] = np.sum(seg)
        self.stats["low level image voxels"] = np.prod(seg.shape)
        # in seg is now stored low resolution segmentation
        # back to normal parameters
        # step 2: discontinuity localization
        # self.segparams = sparams_hi
        seg_border = scipy.ndimage.filters.laplace(seg, mode="constant")
        logger.debug("seg_border: %s", scipy.stats.describe(seg_border, axis=None))
        # logger.debug(str(np.max(seg_border)))
        # logger.debug(str(np.min(seg_border)))
        seg_border[seg_border 

In [6]:
# ============================================================
# CODE BLOCK 6: Prepare multilingual train/eval data
# ============================================================

def safe_select(dataset_obj, start, end):
    """
    Safely selects a slice from a Hugging Face Dataset.

    Why this helper is needed:
    - Python and Java datasets may not have the same size.
    - We should never request indices beyond the dataset length.
    - If start >= end, the function returns None instead of crashing.
    """

    start = min(start, len(dataset_obj))
    end = min(end, len(dataset_obj))

    if start >= end:
        return None

    return dataset_obj.select(range(start, end))


def remove_empty_examples(dataset_obj):
    """
    Removes rows where code or docstring is empty.

    Why this matters:
    - Empty code cannot teach the model anything.
    - Empty docstrings create bad targets.
    - Empty examples can distort training loss and evaluation metrics.
    """

    return dataset_obj.filter(
        lambda example: example["code"].strip() != "" and example["docstring"].strip() != ""
    )


python_dataset = remove_empty_examples(python_dataset)
java_dataset = remove_empty_examples(java_dataset)

print("Python rows after empty filtering:", len(python_dataset))
print("Java rows after empty filtering:", len(java_dataset))


def build_balanced_language_split(
    python_dataset,
    java_dataset,
    python_train_count,
    java_train_count,
    python_eval_count,
    java_eval_count
):
    """
    Builds balanced train/eval splits from Python and Java datasets.

    Why balanced sampling?
    - If Python has many more examples than Java, the model may overfit to Python style.
    - If Java dominates, Python quality may degrade.
    - Balanced sampling gives both languages equal learning opportunity.
    """

    train_parts = []
    eval_parts = []

    python_dataset = python_dataset.shuffle(seed=42)
    java_dataset = java_dataset.shuffle(seed=42)

    python_train = safe_select(
        python_dataset,
        0,
        python_train_count
    )

    python_eval = safe_select(
        python_dataset,
        python_train_count,
        python_train_count + python_eval_count
    )

    java_train = safe_select(
        java_dataset,
        0,
        java_train_count
    )

    java_eval = safe_select(
        java_dataset,
        java_train_count,
        java_train_count + java_eval_count
    )

    if python_train is not None:
        train_parts.append(python_train)

    if java_train is not None:
        train_parts.append(java_train)

    if python_eval is not None:
        eval_parts.append(python_eval)

    if java_eval is not None:
        eval_parts.append(java_eval)

    train_split = concatenate_datasets(train_parts).shuffle(seed=123)
    eval_split = concatenate_datasets(eval_parts).shuffle(seed=456)

    return train_split, eval_split


if DEMO_MODE:
    print("DEMO_MODE=True: using realistic but Colab-friendly multilingual subset.")

    train_data, eval_data = build_balanced_language_split(
        python_dataset=python_dataset,
        java_dataset=java_dataset,
        python_train_count=PYTHON_TRAIN_SAMPLES,
        java_train_count=JAVA_TRAIN_SAMPLES,
        python_eval_count=PYTHON_EVAL_SAMPLES,
        java_eval_count=JAVA_EVAL_SAMPLES
    )

else:
    print("DEMO_MODE=False: preparing data for K-sample chunked training.")

    python_shuffled = python_dataset.shuffle(seed=42)
    java_shuffled = java_dataset.shuffle(seed=42)

    python_eval = safe_select(
        python_shuffled,
        0,
        NON_DEMO_EVAL_PER_LANGUAGE
    )

    java_eval = safe_select(
        java_shuffled,
        0,
        NON_DEMO_EVAL_PER_LANGUAGE
    )

    python_train = safe_select(
        python_shuffled,
        NON_DEMO_EVAL_PER_LANGUAGE,
        len(python_shuffled)
    )

    java_train = safe_select(
        java_shuffled,
        NON_DEMO_EVAL_PER_LANGUAGE,
        len(java_shuffled)
    )

    eval_data = concatenate_datasets(
        [python_eval, java_eval]
    ).shuffle(seed=42)

    train_data = concatenate_datasets(
        [python_train, java_train]
    ).shuffle(seed=42)

print("Train Samples:", len(train_data))
print("Eval Samples:", len(eval_data))

print("\nTrain language distribution:")
print(pd.Series(train_data["language_tag"]).value_counts())

print("\nEval language distribution:")
print(pd.Series(eval_data["language_tag"]).value_counts())

Filter:   0%|          | 0/412178 [00:00<?, ? examples/s]

Filter:   0%|          | 0/454451 [00:00<?, ? examples/s]

Python rows after empty filtering: 412178
Java rows after empty filtering: 454451
DEMO_MODE=True: using realistic but Colab-friendly multilingual subset.
Train Samples: 5000
Eval Samples: 500

Train language distribution:
Java      2500
Python    2500
Name: count, dtype: int64

Eval language distribution:
Java      250
Python    250
Name: count, dtype: int64


In [7]:
# ============================================================
# CODE BLOCK 7: Helper functions with language-aware prompting
# ============================================================

import re
# re is Python's regular expression library.
# We use it to remove HTML/Javadoc tags from Java documentation.

def clean_reference_documentation(doc):
    """
    Cleans reference documentation from the dataset.

    Why this is needed:
    - Python docstrings are usually clean natural language.
    - Java CodeSearchNet examples can contain long Javadoc text.
    - Java Javadocs often include HTML tags like <p>, <a>, <code>.
    - They may also include @param, @return, @throws, @see sections.
    - For Stage 1 documentation generation, we want the main natural-language
      summary, not the entire verbose API documentation block.
    """

    if doc is None:
        return ""

    doc = str(doc)

    # Remove HTML/Javadoc tags such as:
    # <p>, </p>, <a href="...">, </a>, <code>, </code>
    doc = re.sub(r"<[^>]+>", " ", doc)

    # Keep only the main description before structured Javadoc sections.
    stop_markers = [
        "@param",
        "@return",
        "@throws",
        "@sample",
        "@see"
    ]

    for marker in stop_markers:
        if marker in doc:
            doc = doc.split(marker)[0]

    # Normalize whitespace.
    doc = " ".join(doc.split())

    return doc.strip()


def get_code(example):
    """
    Extracts source code from a standardized dataset example.
    """

    return example["code"]


def get_doc(example):
    """
    Extracts and cleans reference documentation.
    """

    return clean_reference_documentation(example["docstring"])


def get_language(example):
    """
    Extracts language label from a standardized dataset example.
    """

    return example["language_tag"]


def build_prompt(code, language):
    """
    Constructs a language-aware instruction prompt.
    """

    prompt = (
        "### Task:\n"
        f"Generate documentation for the following {language} function or method.\n\n"
        "### Language:\n"
        f"{language}\n\n"
        "### Code:\n"
        f"{code}\n\n"
        "### Documentation:\n"
    )

    return prompt

In [8]:
# ============================================================
# CODE BLOCK 8: Load Baseline Model
# ============================================================

from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
# Import classes from Hugging Face Transformers:
# - AutoTokenizer: automatically loads the correct tokenizer for the chosen model.
# - AutoModelForCausalLM: loads a causal language model (predicts next token in sequence).

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME   # Name of the pretrained model defined in CODE BLOCK 3.
                 # Here: "Salesforce/codegen-350M-multi".
                 # Hugging Face will download the tokenizer configuration and vocab.
)

tokenizer.pad_token = tokenizer.eos_token
# Set the padding token to be the same as the end-of-sequence (EOS) token.
# Many causal language models (like CodeGen, GPT) don’t have a dedicated pad token.
# Using EOS ensures consistent padding behavior during batching.

baseline_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME   # Load the pretrained causal language model weights.
                 # "Salesforce/codegen-350M-multi" is a 350M parameter model
                 # trained on multiple programming languages.
)

baseline_model.to(DEVICE)
# Move the model to the selected device (GPU if available, otherwise CPU).
# This ensures training and inference run on the correct hardware.

print("Baseline model loaded.")
# Confirmation message to indicate the model and tokenizer are ready.


tokenizer_config.json:   0%|          | 0.00/240 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/797M [00:00<?, ?B/s]

Baseline model loaded.


In [9]:
# ============================================================
# CODE BLOCK 9: Generated documentation cleaning helper
# ============================================================

def clean_generated_documentation(text):
    """
    Cleans generated model output so only documentation remains.
    """

    if "### Documentation:" in text:
        text = text.split("### Documentation:")[-1]

    stop_markers = [
        "### Task:",
        "### Language:",
        "### Code:",
        "### Inputs:",
        "### Outputs:",
        "### Resource names:",
        "### ID:",
        "def ",
        "\nclass ",
        "\nreturn ",
        "public ",
        "private ",
        "protected ",
        "static ",
        "@Override",
        "@param",
        "@return",
        "@throws",
        "@sample",
        "@see",
    ]

    for marker in stop_markers:
        if marker in text:
            text = text.split(marker)[0]

    # Remove HTML tags.
    text = re.sub(r"<[^>]+>", " ", text)

    # Remove docstring/Javadoc formatting artifacts.
    text = text.replace('"""', "")
    text = text.replace("/**", "")
    text = text.replace("*/", "")
    text = text.replace("*", "")

    # Normalize spacing.
    text = " ".join(text.split())

    return text.strip()

In [10]:
# ============================================================
# CODE BLOCK 10: Baseline generation helper
# ============================================================

def generate_documentation(model, code, language):
    """
    Generates documentation for either Python or Java code.
    """

    # Build the instruction-style prompt that includes the code and language tag.
    # This prompt tells the model what task to perform (generate documentation).
    prompt = build_prompt(code, language)

    # Tokenize the prompt into input IDs (numerical tokens).
    # - return_tensors="pt" → returns PyTorch tensors.
    # - truncation=True → cuts off text if it exceeds MAX_LENGTH.
    # - max_length=MAX_LENGTH → ensures consistent input size.
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    )

    # Move all tensors (input_ids, attention_mask) to the correct device (CPU/GPU).
    inputs = {
        key: value.to(DEVICE)
        for key, value in inputs.items()
    }

    # Record the length of the input prompt in tokens.
    # This helps us later separate prompt tokens from generated tokens.
    input_length = inputs["input_ids"].shape[1]

    # Put the model in evaluation mode (no dropout, stable inference).
    model.eval()

    # Disable gradient calculations for inference (saves memory, faster).
    with torch.no_grad():
        # Generate new tokens from the model.
        # - max_new_tokens=80 → limit docstring length.
        # - do_sample=False → greedy decoding (deterministic, always pick highest probability).
        # - pad_token_id=tokenizer.eos_token_id → ensures padding handled correctly.
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Slice off the prompt tokens, keeping only the newly generated documentation tokens.
    generated_tokens = outputs[0][input_length:]

    # Decode token IDs back into human-readable text.
    # - skip_special_tokens=True → removes <pad>, <eos>, etc.
    raw_generated = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    # Clean the raw text (remove leakage, extra markers) and return final docstring.
    return clean_generated_documentation(raw_generated)


In [11]:
# ============================================================
# CODE BLOCK 11: Baseline example
# ============================================================

sample = eval_data[0]

code = get_code(sample)

reference = get_doc(sample)

language = get_language(sample)

baseline_output = generate_documentation(
    baseline_model,
    code,
    language
)

print("LANGUAGE:\n")
print(language)

print("\nREFERENCE AFTER CLEANING:\n")
print(reference)

print("\nMODEL OUTPUT AFTER CLEANING:\n")
print(baseline_output)

LANGUAGE:

Java

REFERENCE AFTER CLEANING:

Uploads a server certificate entity for the AWS account. The server certificate entity includes a public key certificate, a private key, and an optional certificate chain, which should all be PEM-encoded. We recommend that you use AWS Certificate Manager to provision, manage, and deploy your server certificates. With ACM you can request a certificate, deploy it to AWS resources, and let ACM handle certificate renewals for you. Certificates provided by ACM are free. For more information about using ACM, see the AWS Certificate Manager User Guide . For more information about working with server certificates, see Working with Server Certificates in the IAM User Guide . This topic includes a list of AWS services that can use the server certificates that you manage with IAM. For information about the number of server certificates you can upload, see Limitations on IAM Entities and Objects in the IAM User Guide . Because the body of the public key 

In [12]:
# ============================================================
# CODE BLOCK 12: Metrics
# ============================================================

import evaluate
# Hugging Face Evaluate library.
# We use it to compute ROUGE-L for documentation similarity.

rouge = evaluate.load("rouge")
# ROUGE-L measures lexical/sequence overlap between generated documentation
# and reference documentation.

In [13]:
# ============================================================
# CODE BLOCK 14: ROUGE-L evaluation function
# ============================================================

def rouge_l_score(prediction, reference):
    """
    Computes ROUGE-L between generated documentation and reference documentation.
    """

    result = rouge.compute(
        predictions=[prediction],
        references=[reference]
    )

    return result["rougeL"]

In [17]:
# ============================================================
# CODE BLOCK 15: CodeBERTScore with length filtering
# ============================================================

def should_compute_codebert_score(prediction, reference):
    """
    Decides whether CodeBERTScore should be computed.

    Why:
    - Empty outputs should not be embedded (they produce misleading vectors).
    - Very short outputs can give artificially high similarity scores.
    - Very long outputs can slow evaluation and dilute semantic meaning.
    """

    # Split prediction and reference into word lists.
    pred_words = prediction.split()
    ref_words = reference.split()

    # If prediction is too short, skip scoring.
    if len(pred_words) < MIN_DOC_WORDS_FOR_CODEBERT:
        return False

    # If reference is too short, skip scoring.
    if len(ref_words) < MIN_DOC_WORDS_FOR_CODEBERT:
        return False

    # If prediction is too long, skip scoring.
    if len(pred_words) > MAX_DOC_WORDS_FOR_CODEBERT:
        return False

    # If reference is too long, skip scoring.
    if len(ref_words) > MAX_DOC_WORDS_FOR_CODEBERT:
        return False

    # Otherwise, safe to compute CodeBERTScore.
    return True


def codebert_score(prediction, reference):
    """
    Computes semantic similarity using official BERTScore
    with microsoft/codebert-base as the encoder.
    """

    if not should_compute_codebert_score(prediction, reference):
        return None

    P, R, F1 = bert_score(
        [prediction],
        [reference],
        model_type="microsoft/codebert-base",
        num_layers=12,
        lang="en",
        device=DEVICE,
        verbose=False
    )

    return float(F1.item())

In [18]:
# ============================================================
# CODE BLOCK 16: Full Stage 1 automatic evaluation
# ============================================================

def evaluate_prediction(prediction, reference):
    """
    Evaluates generated documentation.

    ROUGE-L:
    - Always computed.
    - Empty prediction naturally receives low/zero score.

    CodeBERTScore:
    - Computed only if prediction/reference pass length filter.
    - Otherwise returned as None.
    """

    rouge_score = rouge_l_score(
        prediction,
        reference
    )

    semantic_score = codebert_score(
        prediction,
        reference
    )

    return {
        "ROUGE-L": round(rouge_score, 4),
        "CodeBERTScore": round(semantic_score, 4) if semantic_score is not None else None
    }

In [19]:
# ============================================================
# CODE BLOCK 17: Baseline Evaluation
# ============================================================

TEST_EXAMPLES = min(TEST_EXAMPLES, len(eval_data))

print(f"Running baseline evaluation on {TEST_EXAMPLES} examples...")

baseline_results = []

for i in range(TEST_EXAMPLES):

    example = eval_data[i]

    code = get_code(example)

    reference = get_doc(example)

    language = get_language(example)

    try:
        prediction = generate_documentation(
            baseline_model,
            code,
            language
        )

        # Strict cleaner may return an empty string.
        # We keep that scientifically valid, but label it for readability.
        if prediction.strip() == "":
            prediction = "EMPTY_OUTPUT"

        metrics = evaluate_prediction(
            prediction,
            reference
        )

        baseline_results.append({
            "index": i,
            "language": language,
            "reference": reference,
            "baseline_prediction": prediction,
            "baseline_ROUGE_L": metrics["ROUGE-L"],
            "baseline_CodeBERTScore": metrics["CodeBERTScore"]
        })

    except Exception as error:
        print(f"Error on sample {i}: {error}")

baseline_df = pd.DataFrame(baseline_results)

print("\nCompleted baseline evaluation.")
print("Number of successful evaluations:", len(baseline_df))

baseline_df.head()

Running baseline evaluation on 100 examples...


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]


Completed baseline evaluation.
Number of successful evaluations: 100


,index,language,reference,baseline_prediction,baseline_ROUGE_L,baseline_CodeBERTScore
0,0,Java,Uploads a server certificate entity for the AW...,EMPTY_OUTPUT,0.0,NaN
1,1,Python,:param api: :param str vcenter_name: :rtype: V...,EMPTY_OUTPUT,0.0,NaN
2,2,Python,spectrum of correlation matrix and largest cor...,")') axis([-1, 1]) axis([-1, 1]) axis([-1, 1]) ...",0.0,0.7856
3,3,Java,The notification configurations. NOTE: This me...,### Example:,0.0,NaN
4,4,Python,performs all possible permutations of route im...,EMPTY_OUTPUT,0.0,NaN


In [20]:
# ============================================================
# CODE BLOCK 18: Baseline Metrics Summary
# ============================================================

baseline_avg_rouge = baseline_df["baseline_ROUGE_L"].mean()

baseline_avg_codebert = baseline_df[
    "baseline_CodeBERTScore"
].dropna().mean()

print("=" * 50)
print("BASELINE MODEL PERFORMANCE")
print("=" * 50)

print("Average ROUGE-L:", round(baseline_avg_rouge, 4))
print("Average CodeBERTScore:", round(baseline_avg_codebert, 4))

print("\nBaseline performance by language:")
display(
    baseline_df.groupby("language")[
        ["baseline_ROUGE_L", "baseline_CodeBERTScore"]
    ].mean()
)

print("=" * 50)

BASELINE_ROUGE = baseline_avg_rouge
BASELINE_CODEBERT = baseline_avg_codebert

# ------------------------------------------------------------
# Empty Output Rate
# ------------------------------------------------------------
# Measures how often the model completely failed to generate
# usable documentation.

baseline_empty_outputs = (
    baseline_df["baseline_prediction"] == "EMPTY_OUTPUT"
).sum()

baseline_empty_rate = (
    baseline_empty_outputs / len(baseline_df)
) * 100

print(f"Baseline Empty Output Rate: {baseline_empty_rate:.2f}%")
print(f"Empty Outputs: {baseline_empty_outputs}/{len(baseline_df)}")

BASELINE MODEL PERFORMANCE
Average ROUGE-L: 0.0172
Average CodeBERTScore: 0.7833

Baseline performance by language:


,baseline_ROUGE_L,baseline_CodeBERTScore
language,,
Java,0.018578,0.775659
Python,0.016158,0.801767


Baseline Empty Output Rate: 47.00%
Empty Outputs: 47/100


In [21]:
# ============================================================
# CODE BLOCK 19: Tokenization with docstring-only labels
# ============================================================

def build_prompt_without_answer(example):
    """
    Builds only the input/context portion of the prompt.

    This includes:
    - task instruction
    - programming language
    - source code
    - documentation marker

    It does NOT include the target documentation.

    Why:
    During training, we want the model to use this part as context.
    We do not want to compute loss on this prompt section.
    """

    code = get_code(example)

    language = get_language(example)

    prompt = (
        "### Task:\n"
        f"Generate documentation for the following {language} function or method.\n\n"
        "### Language:\n"
        f"{language}\n\n"
        "### Code:\n"
        f"{code}\n\n"
        "### Documentation:\n"
    )

    return prompt


def build_answer_text(example):
    """
    Builds the target answer portion.

    This is the cleaned reference documentation/docstring.

    Why:
    This is the only part we want the model to learn to generate.
    """

    doc = get_doc(example)

    return doc


def tokenize_for_training(example):
    """
    Tokenizes training examples for causal language model fine-tuning.

    Correct training logic:

    Input sequence:
        prompt + answer

    Labels:
        -100 for prompt tokens
        actual token IDs for answer tokens
        -100 for padding tokens

    Why this is better:
    - The model sees task + language + code as context.
    - Loss is computed only on the documentation.
    - The model is not punished for failing to reproduce the prompt/code.
    - This aligns training with inference, where we provide the prompt and
      expect the model to generate only documentation.
    """

    prompt_text = build_prompt_without_answer(example)

    answer_text = build_answer_text(example)

    full_text = prompt_text + answer_text

    tokenized_full = tokenizer(
        full_text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

    tokenized_prompt = tokenizer(
        prompt_text,
        truncation=True,
        padding=False,
        max_length=MAX_LENGTH
    )

    input_ids = tokenized_full["input_ids"]

    attention_mask = tokenized_full["attention_mask"]

    labels = input_ids.copy()

    prompt_length = len(tokenized_prompt["input_ids"])

    labels = [
        token_id if index >= prompt_length and mask == 1 else -100
        for index, (token_id, mask) in enumerate(
            zip(input_ids, attention_mask)
        )
    ]

    tokenized_full["labels"] = labels

    return tokenized_full


if DEMO_MODE:
    tokenized_train = train_data.map(
        tokenize_for_training,
        remove_columns=train_data.column_names
    )

else:
    # In non-demo mode, we avoid tokenizing the full large dataset here.
    # K-sample chunks are tokenized inside Block 23.
    tokenized_train = None


tokenized_eval = eval_data.map(
    tokenize_for_training,
    remove_columns=eval_data.column_names
)

print("Tokenization completed with docstring-only labels.")
print("Eval keys:", tokenized_eval[0].keys())

if tokenized_train is not None:
    print("Train keys:", tokenized_train[0].keys())


# ------------------------------------------------------------
# Sanity check: verify that prompt tokens are masked
# ------------------------------------------------------------

sample_labels = tokenized_eval[0]["labels"]

num_supervised_tokens = sum(
    1 for label in sample_labels if label != -100
)

num_masked_tokens = sum(
    1 for label in sample_labels if label == -100
)

print("Supervised documentation tokens:", num_supervised_tokens)

print("Masked prompt/padding tokens:", num_masked_tokens)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenization completed with docstring-only labels.
Eval keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
Train keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
Supervised documentation tokens: 275
Masked prompt/padding tokens: 237


In [22]:
# ============================================================
# CODE BLOCK 20: Training Example Inspection
# ============================================================

"""
Purpose:
--------
This block is NOT used for training.

It is purely for:
1. Understanding what one training sample looks like.
2. Debugging prompt construction.
3. Demonstrating the training setup during mentor reviews.
4. Verifying that the documentation cleaning pipeline works correctly.

Why keep this block?
--------------------
we can show:

Raw code
Reference documentation
Final prompt used for training

This makes the training pipeline transparent and easy to understand.
"""

sample = train_data[0]

print("=" * 80)
print("LANGUAGE")
print("=" * 80)

print(get_language(sample))

print("\n")

print("=" * 80)
print("CODE")
print("=" * 80)

print(get_code(sample)[:1500])

print("\n")

print("=" * 80)
print("REFERENCE DOCUMENTATION (AFTER CLEANING)")
print("=" * 80)

print(get_doc(sample)[:1000])

print("\n")

print("=" * 80)
print("PROMPT PROVIDED TO MODEL")
print("=" * 80)

print(
    build_prompt_without_answer(sample)
)

print("\n")

print("=" * 80)
print("TARGET DOCUMENTATION")
print("=" * 80)

print(
    build_answer_text(sample)
)

print("\n")

print("=" * 80)
print("TRAINING OBJECTIVE")
print("=" * 80)

print(
    "Model sees prompt as context.\n"
    "Loss is computed ONLY on the target documentation.\n"
    "Prompt tokens are masked using -100."
)

LANGUAGE
Java


CODE
public static void scrollToRow(JTable table, int row)
    {
        Rectangle visibleRect = table.getVisibleRect();
        Rectangle cellRect = table.getCellRect(row, 0, true);
        Rectangle r = new Rectangle(
            visibleRect.x, cellRect.y, 
            visibleRect.width, cellRect.height);
        table.scrollRectToVisible(r);
    }


REFERENCE DOCUMENTATION (AFTER CLEANING)
Scroll the given table so that the specified row is visible.


PROMPT PROVIDED TO MODEL
### Task:
Generate documentation for the following Java function or method.

### Language:
Java

### Code:
public static void scrollToRow(JTable table, int row)
    {
        Rectangle visibleRect = table.getVisibleRect();
        Rectangle cellRect = table.getCellRect(row, 0, true);
        Rectangle r = new Rectangle(
            visibleRect.x, cellRect.y, 
            visibleRect.width, cellRect.height);
        table.scrollRectToVisible(r);
    }

### Documentation:



TARGET DOCUMENTATION
S

In [23]:
# ============================================================
# CODE BLOCK 21: Load fresh model and attach LoRA adapters
# ============================================================

try:
    # Delete the previously loaded baseline model (if it exists).
    # This frees up GPU memory before loading a new model.
    del baseline_model
    torch.cuda.empty_cache()  # Clear CUDA cache to avoid OOM errors.
except:
    # If baseline_model was never defined, just skip without crashing.
    pass

# Load the pretrained causal language model (e.g., CodeGen).
# AutoModelForCausalLM automatically selects the right architecture.
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Move the model to the correct device (GPU if available, else CPU).
base_model.to(DEVICE)

# Configure LoRA (Low-Rank Adaptation) adapters for parameter-efficient fine-tuning.
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,   # Task type: causal language modeling.
    r=8,                            # Rank of low-rank matrices (controls adapter size).
    lora_alpha=16,                  # Scaling factor for LoRA updates.
    lora_dropout=0.05,              # Dropout for regularization (prevents overfitting).
    target_modules=[
        "qkv_proj",                 # Adapt attention query/key/value projection layers.
        "out_proj"                  # Adapt attention output projection layer.
    ],
    bias="none"                     # Do not train bias terms.
)

# Wrap the base model with LoRA adapters.
# This injects small trainable matrices into the specified layers.
finetune_model = get_peft_model(
    base_model,
    lora_config
)

# Print how many parameters are trainable vs frozen.
# Useful sanity check to confirm LoRA is active.
finetune_model.print_trainable_parameters()

# Confirmation message.
print("Fresh CodeGen model loaded with LoRA adapters.")


trainable params: 983,040 || all params: 357,695,488 || trainable%: 0.2748
Fresh CodeGen model loaded with LoRA adapters.


In [24]:
# ============================================================
# CODE BLOCK 22: LoRA fine-tuning setup
# ============================================================

# If an output directory already exists (from a previous run),
# remove it to ensure a clean slate for saving checkpoints/logs.
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

# Define training arguments for Hugging Face Trainer.
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,              # Directory where checkpoints and logs will be saved.

    num_train_epochs=1,                 # Number of full passes through the training dataset.
                                        # Demo run uses 1 epoch for speed.

    per_device_train_batch_size=1,      # Batch size per device during training.
    per_device_eval_batch_size=1,       # Batch size per device during evaluation.
                                        # Small batch sizes reduce GPU memory usage.

    gradient_accumulation_steps=8,      # Accumulate gradients over 8 steps before updating weights.
                                        # Effective batch size = 1 * 8 = 8.

    learning_rate=2e-4,                 # Learning rate for optimizer (slightly higher for LoRA fine-tuning).

    logging_steps=20,                   # Log training metrics every 20 steps.

    save_steps=100,                     # Save a checkpoint every 100 steps.

    evaluation_strategy="no",           # Disable automatic evaluation during training.
                                        # Evaluation can be run manually after training.

    fp16=False,                         # Disable mixed precision training.
                                        # Set to True if GPU supports FP16 for faster training.

    report_to="none"                    # Disable external logging integrations (e.g., WandB, TensorBoard).
)

# In demo mode, we create the Trainer immediately using the small tokenized dataset.
if DEMO_MODE:
    trainer = Trainer(
        model=finetune_model,           # LoRA-wrapped model to fine-tune.
        args=training_args,             # Training configuration defined above.
        train_dataset=tokenized_train,  # Tokenized training dataset (small demo subset).
        eval_dataset=tokenized_eval,    # Tokenized evaluation dataset.
        data_collator=default_data_collator # Handles dynamic padding and batching.
    )

    print("LoRA Trainer ready for demo mode.")

else:
    # In non-demo mode, we avoid creating a Trainer here.
    # Instead, Trainer will be created later per K-sample chunk (Block 23).
    trainer = None

    print("Non-demo mode: Trainer will be created per K-sample chunk.")


LoRA Trainer ready for demo mode.


/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [25]:
# ============================================================
# CODE BLOCK 23: LoRA fine-tuning with optional K-sample chunked training
# ============================================================

def select_language_subset(dataset_obj, language_name):
    """
    Filters dataset by language.

    Purpose:
    - Ensures balanced training by splitting dataset into language-specific pools.
    - Example: separate Python vs Java samples.
    """
    return dataset_obj.filter(
        lambda example: example["language_tag"] == language_name
    )


def tokenize_chunk(raw_chunk):
    """
    Tokenizes one K-sample chunk.

    Why:
    - Tokenization builds prompts internally (task + code + docstring).
    - We can directly map raw dataset examples into model-ready format.
    """
    tokenized_chunk = raw_chunk.map(
        tokenize_for_training,              # Function from Block 20.
        remove_columns=raw_chunk.column_names  # Drop original columns, keep only model inputs.
    )
    return tokenized_chunk


def train_one_chunk(raw_chunk, round_id):
    """
    Trains LoRA adapters on one K-sample chunk.

    Steps:
    - Tokenize the chunk.
    - Create a Trainer for this chunk.
    - Run training.
    - Save checkpoint for this round.
    """

    print(f"\nPreparing training chunk for round {round_id}...")

    # Tokenize the raw chunk into input_ids + labels.
    tokenized_chunk = tokenize_chunk(raw_chunk)

    # Create a Trainer instance for this chunk.
    chunk_trainer = Trainer(
        model=finetune_model,              # LoRA-wrapped model.
        args=training_args,                # Training configuration.
        train_dataset=tokenized_chunk,     # Current chunk as training data.
        eval_dataset=tokenized_eval,       # Evaluation dataset (fixed).
        data_collator=default_data_collator # Handles padding/batching.
    )

    print(f"Starting training round {round_id} on {len(tokenized_chunk)} samples...")

    # Train on this chunk.
    chunk_trainer.train()

    # Save checkpoint for this round.
    checkpoint_path = f"{OUTPUT_DIR}/lora_round_{round_id}"

    finetune_model.save_pretrained(checkpoint_path)
    tokenizer.save_pretrained(checkpoint_path)

    print(f"Saved LoRA checkpoint for round {round_id} at: {checkpoint_path}")


# --- Main training loop ---
if DEMO_MODE:
    # In demo mode, train once on the small multilingual demo subset.
    print("DEMO_MODE=True: training once on multilingual demo subset.")
    trainer.train()

else:
    # In full mode, use balanced K-sample chunked training.
    print("DEMO_MODE=False: using balanced K-sample chunked training.")

    # Split dataset into language-specific pools.
    python_train_pool = select_language_subset(train_data, "Python")
    java_train_pool = select_language_subset(train_data, "Java")

    # Iterate over training rounds.
    for round_id in range(1, K_TRAINING_ROUNDS + 1):

        round_parts = []  # Collect chunks for this round.

        # --- Python chunk selection ---
        py_start = (round_id - 1) * K_SAMPLES_PER_LANGUAGE_PER_ROUND
        py_end = py_start + K_SAMPLES_PER_LANGUAGE_PER_ROUND

        py_chunk = safe_select(
            python_train_pool,
            py_start,
            py_end
        )

        if py_chunk is not None:
            round_parts.append(py_chunk)

        # --- Java chunk selection ---
        java_start = (round_id - 1) * K_SAMPLES_PER_LANGUAGE_PER_ROUND
        java_end = java_start + K_SAMPLES_PER_LANGUAGE_PER_ROUND

        java_chunk = safe_select(
            java_train_pool,
            java_start,
            java_end
        )

        if java_chunk is not None:
            round_parts.append(java_chunk)

        # If no samples left, stop training.
        if len(round_parts) == 0:
            print(f"No more samples available for round {round_id}. Stopping.")
            break

        # Concatenate Python + Java chunks for this round.
        raw_round_chunk = concatenate_datasets(round_parts).shuffle(
            seed=42 + round_id  # Shuffle with round-specific seed for reproducibility.
        )

        print(f"Round {round_id}: training on {len(raw_round_chunk)} samples.")
        print(pd.Series(raw_round_chunk["language_tag"]).value_counts())

        # Train on this round’s chunk.
        train_one_chunk(raw_round_chunk, round_id)

# --- Final save after all rounds ---
print("Fine-tuning completed.")

finetune_model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Final LoRA adapter saved to:", OUTPUT_DIR)


DEMO_MODE=True: training once on multilingual demo subset.


Step,Training Loss
20,3.653900
40,4.558600
60,4.366200
80,2.774300
100,2.569200
120,3.274000
140,6.669100
160,4.442000
180,2.214700
200,2.463700


Fine-tuning completed.
Final LoRA adapter saved to: /content/stage1_codegen_doc_lora_model


In [26]:
# ============================================================
# CODE BLOCK 24: Fine-tuned multilingual model evaluation
# ============================================================

finetuned_results = []

for i in range(TEST_EXAMPLES):

    example = eval_data[i]

    code = get_code(example)

    reference = get_doc(example)

    language = get_language(example)

    prediction = generate_documentation(
        finetune_model,
        code,
        language
    )

    if prediction.strip() == "":
        prediction = "EMPTY_OUTPUT"

    metrics = evaluate_prediction(
        prediction,
        reference
    )

    finetuned_results.append({
        "index": i,
        "language": language,
        "finetuned_prediction": prediction,
        "finetuned_ROUGE_L": metrics["ROUGE-L"],
        "finetuned_CodeBERTScore": metrics["CodeBERTScore"]
    })

finetuned_df = pd.DataFrame(finetuned_results)

finetuned_df.head()

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


,index,language,finetuned_prediction,finetuned_ROUGE_L,finetuned_CodeBERTScore
0,0,Java,Uploads a server certificate. This operation c...,0.1915,0.8317
1,1,Python,:param api: :param str vcenter_name: :rtype: V...,0.5333,0.9327
2,2,Python,")') hold(True) semilogy(x, y, '-c') hold(True)...",0.0000,0.8076
3,3,Java,The notification configurations. Each notifica...,0.1379,0.7897
4,4,Python,performs all possible permutations of route im...,0.5397,0.9315


In [27]:
# ============================================================
# CODE BLOCK 25: Fine-tuned average scores
# ============================================================

finetuned_avg_rouge = finetuned_df["finetuned_ROUGE_L"].mean()

finetuned_avg_codebert = finetuned_df[
    "finetuned_CodeBERTScore"
].dropna().mean()

# ------------------------------------------------------------
# Empty Output Rate
# ------------------------------------------------------------
# Measures how often the fine-tuned model failed to generate
# usable documentation.

finetuned_empty_outputs = (
    finetuned_df["finetuned_prediction"] == "EMPTY_OUTPUT"
).sum()

finetuned_empty_rate = (
    finetuned_empty_outputs / len(finetuned_df)
) * 100

print(f"Fine-Tuned Empty Output Rate: {finetuned_empty_rate:.2f}%")
print(
    f"Empty Outputs: "
    f"{finetuned_empty_outputs}/{len(finetuned_df)}"
)

print("Fine-Tuned Average ROUGE-L:", round(finetuned_avg_rouge, 4))
print("Fine-Tuned Average CodeBERTScore:", round(finetuned_avg_codebert, 4))

print("\nFine-tuned performance by language:")
display(
    finetuned_df.groupby("language")[
        ["finetuned_ROUGE_L", "finetuned_CodeBERTScore"]
    ].mean()
)

Fine-Tuned Empty Output Rate: 0.00%
Empty Outputs: 0/100
Fine-Tuned Average ROUGE-L: 0.2997
Fine-Tuned Average CodeBERTScore: 0.8776

Fine-tuned performance by language:


,finetuned_ROUGE_L,finetuned_CodeBERTScore
language,,
Java,0.097980,0.827700
Python,0.464811,0.905792


In [28]:
# ============================================================
# CODE BLOCK 26: Comparison table
# ============================================================

comparison_df = pd.DataFrame([
    {
        "Model": "Baseline",
        "ROUGE-L": round(baseline_avg_rouge, 4),
        "CodeBERTScore": round(baseline_avg_codebert, 4),
        "Empty Output Rate (%)": round(
            baseline_empty_rate,
            2
        )
    },
    {
        "Model": "LoRA Fine-Tuned",
        "ROUGE-L": round(finetuned_avg_rouge, 4),
        "CodeBERTScore": round(finetuned_avg_codebert, 4),
        "Empty Output Rate (%)": round(
            finetuned_empty_rate,
            2
        )
    }
])

comparison_df

,Model,ROUGE-L,CodeBERTScore,Empty Output Rate (%)
0,Baseline,0.0172,0.7833,47.0
1,LoRA Fine-Tuned,0.2997,0.8776,0.0


In [29]:
# ============================================================
# CODE BLOCK 27: Example output comparison
# ============================================================

example_comparison_df = baseline_df.merge(
    finetuned_df,
    on=["index", "language"]
)

example_comparison_df[
    [
        "index",
        "language",
        "reference",
        "baseline_prediction",
        "finetuned_prediction",
        "baseline_ROUGE_L",
        "finetuned_ROUGE_L",
        "baseline_CodeBERTScore",
        "finetuned_CodeBERTScore"
    ]
].head()

,index,language,reference,baseline_prediction,finetuned_prediction,baseline_ROUGE_L,finetuned_ROUGE_L,baseline_CodeBERTScore,finetuned_CodeBERTScore
0,0,Java,Uploads a server certificate entity for the AW...,EMPTY_OUTPUT,Uploads a server certificate. This operation c...,0.0,0.1915,NaN,0.8317
1,1,Python,:param api: :param str vcenter_name: :rtype: V...,EMPTY_OUTPUT,:param api: :param str vcenter_name: :rtype: V...,0.0,0.5333,NaN,0.9327
2,2,Python,spectrum of correlation matrix and largest cor...,")') axis([-1, 1]) axis([-1, 1]) axis([-1, 1]) ...",")') hold(True) semilogy(x, y, '-c') hold(True)...",0.0,0.0000,0.7856,0.8076
3,3,Java,The notification configurations. NOTE: This me...,### Example:,The notification configurations. Each notifica...,0.0,0.1379,NaN,0.7897
4,4,Python,performs all possible permutations of route im...,EMPTY_OUTPUT,performs all possible permutations of route im...,0.0,0.5397,NaN,0.9315


In [ ]:
# ============================================================
# CODE BLOCK 28: Human evaluation template
# ============================================================

human_eval_template = example_comparison_df[
    [
        "index",
        "language",
        "reference",
        "baseline_prediction",
        "finetuned_prediction"
    ]
].copy()

human_eval_template["Correctness_1_to_5"] = ""
human_eval_template["Completeness_1_to_5"] = ""
human_eval_template["Readability_1_to_5"] = ""
human_eval_template["Notes"] = ""

human_eval_template.head()

,index,reference,baseline_prediction,finetuned_prediction,Correctness_1_to_5,Completeness_1_to_5,Readability_1_to_5,Notes
0,0,apply the function to my values; return a bloc...,,apply the function to my values; return a bloc...,,,,
1,1,fillna on the block with the value. If we fail...,".apply(f, axis=1)",fillna on the block with the value. If we fail...,,,,
2,2,"split the block per-column, and apply the call...",new_blocks.append(block) return new_blocks,"split the block per-column, and apply the call...",,,,
3,3,try to downcast each item to the dict of dtype...,,try to downcast each item to the dict of dtype...,,,,
4,4,Coerce to the new type\n\n Parameters\n...,"kwargs.get('ordered', None) if categories is n...","kwargs.get('ordered', None) if categories is n...",,,,


In [ ]:
# ============================================================
# CODE BLOCK 29: Save evaluation results
# ============================================================

baseline_df.to_csv(
    "/content/stage1_baseline_results.csv",
    index=False
)

finetuned_df.to_csv(
    "/content/stage1_finetuned_results.csv",
    index=False
)

comparison_df.to_csv(
    "/content/stage1_comparison_results.csv",
    index=False
)

human_eval_template.to_csv(
    "/content/stage1_human_eval_template.csv",
    index=False
)

print("Evaluation files saved.")

Evaluation files saved.


In [ ]:
# ============================================================
# CODE BLOCK 30: Colab UI for multilingual Stage 1
# ============================================================

import gradio as gr

def ui_generate_documentation(language, code):
    """
    UI wrapper for multilingual documentation generation.
    """

    output = generate_documentation(
        finetune_model,
        code,
        language
    )

    if output.strip() == "":
        output = "EMPTY_OUTPUT"

    return output


example_python_code = """
def calculate_area(radius):
    return 3.14159 * radius * radius
"""

demo = gr.Interface(
    fn=ui_generate_documentation,

    inputs=[
        gr.Dropdown(
            choices=["Python", "Java"],
            value="Python",
            label="Programming Language"
        ),

        gr.Code(
            label="Enter Code",
            language="python",
            value=example_python_code
        )
    ],

    outputs=gr.Textbox(
        label="Generated Documentation",
        lines=6
    ),

    title="RepoCoder Studio - Stage 1 Multilingual Documentation Generator",

    description=(
        "Paste a Python function or Java method and the LoRA fine-tuned "
        "multilingual CodeGen model will generate documentation."
    )
)

demo.launch(
    share=True,
    debug=True
)